# Imported portfolio

Reads a broker export straight into a `Portfolio` and shows how it has done.
Everything below comes from the import plus `PortfolioSummary.get_summary()`.

Put your CSV in `private/` (gitignored — broker exports are personal records and
this repo is public), or point `HOLDINGS_CSV` at it.

> **Clear outputs before committing.** The tables below contain real position
> sizes and cost bases. This repository is public, and saved cell outputs are
> part of the file. `nbstripout` handles it automatically.


In [ ]:
# Pick up edits to the package without restarting the kernel.
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from data.investing_csv import build_portfolio
from data.market_data import MarketDataFetcher
from reporting.summary import PortfolioSummary

pd.options.display.float_format = '{:,.2f}'.format

CSV_PATH  = os.environ.get('HOLDINGS_CSV', os.path.join('..', 'private', 'holdings.csv'))
BASE      = 'USD'          # everything is reported in this currency
BENCHMARK = 'SPY'

portfolio, report = build_portfolio(CSV_PATH, name='imported', base_currency=BASE)
print(report.summary())

## 1 · What the import made of the file

Rejected rows mean a position is missing, so they are worth reading. Warnings
are usually symbols with no Yahoo listing — delisted, or fund codes that were
never there.

In [ ]:
print(f"open positions   {len(portfolio.open_positions())}")
print(f"total positions  {len(portfolio.all_positions())}   (the rest are closed)")
print(f"transactions     {len(report.transactions)}")
print(f"rejected rows    {len(report.rejected)}")

if report.rejected:
    display(pd.DataFrame(
        [(n, why) for n, why, _ in report.rejected], columns=['line', 'reason']
    ))

## 2 · Current holdings

In [ ]:
fetcher = MarketDataFetcher()
summary = PortfolioSummary(portfolio, fetcher=fetcher, benchmark_ticker=BENCHMARK).get_summary()

v = summary['value_summary']
print(f"Value now        {v['Current Value']:>12,.2f} {BASE}")
print(f"Contributed      {v['Total Invested']:>12,.2f} {BASE}")
print(f"Profit           {v['Total Profit']:>12,.2f} {BASE}")

positions = pd.DataFrame(summary['positions']).sort_values('weight', ascending=False)
positions[['ticker', 'currency', 'quantity', 'avg_cost', 'current_price',
           'current_value', 'unrealized_pnl', 'unrealized_pnl_pct', 'weight']]

`avg_cost` and `current_price` are in the position's own currency; everything
that gets summed or weighted is converted to the base currency first.

In [ ]:
# Where the money sits, by currency.
by_ccy = (pd.DataFrame(summary['positions'])
            .groupby('currency')['current_value'].sum()
            .sort_values(ascending=False))
(by_ccy / by_ccy.sum() * 100).round(1).to_frame(f'% of portfolio ({BASE})')

## 3 · Portfolio value over time

Steps up when money goes in, drifts with the market in between.

In [ ]:
value = summary['timeseries']['portfolio_value']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(value.index, value.values, linewidth=1.4, color='steelblue')
ax.set_title(f'Portfolio value ({BASE})')
ax.set_ylabel(BASE)
ax.grid(alpha=0.3)
plt.show()

## 4 · Performance vs benchmark

Both start at 100. This ignores deposits — it only moves when the market does,
which is what makes it comparable.

In [ ]:
port  = summary['timeseries']['growth_of_1usd']
bench = summary['benchmark']['growth_of_1usd_benchmark'].reindex(port.index).ffill()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(port.index,  port.values,  linewidth=1.4, label='Portfolio', color='steelblue')
ax.plot(bench.index, bench.values, linewidth=1.4, label=BENCHMARK,  color='grey')
ax.axhline(100, color='black', linewidth=0.8, alpha=0.4)
ax.set_title(f'Growth of 100 — portfolio vs {BENCHMARK}')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 5 · The numbers

**Time-weighted** grades the picks: it strips out when money arrived.
**Money-weighted** grades your timing: it is the IRR on what you actually paid
in. A big gap between them means the capital showed up at good or bad moments,
not that either figure is wrong.

In [ ]:
m = summary['performance_metrics']
b = summary['benchmark']

pd.Series({
    'Time-weighted return':      m['Time Weighted Return'],
    'Money-weighted return':     m['Money Weighted return'],
    f'{BENCHMARK} return':       b['benchmark_return'],
    'Outperformance':            b['outperformance'],
    'Annualized':                m['Annualized Return'],
    'Volatility':                m['Volatility'],
    'Sharpe ratio':              m['Sharpe Ratio'],
    'Max drawdown':              m['Max Drawdown'],
    'Beta':                      b['beta'],
    'Correlation':               b['correlation'],
}).to_frame('value')

## 6 · Returns by period

In [ ]:
pd.Series(summary['period_returns']).to_frame('return')

## 7 · Next step — news on these holdings

`holdings` is the ticker list for the Yahoo Finance news lookup, whose headlines
then go to the local LLM. `weights` says which of them actually matter.

In [ ]:
holdings = [p['ticker'] for p in summary['positions']]
weights  = {p['ticker']: round(p['weight'], 4) for p in summary['positions']}
print(holdings)
weights